# Modules import

In [ ]:
# For online projects files edits and no kernel and modules reloaded
%load_ext autoreload
%autoreload 2

In [ ]:
# %reload_ext autoreload
# %pip install -U pip

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

import megatron.config as config

test = pd.read_csv(
    filepath_or_buffer="../data/raw/test.csv",
    parse_dates=["date"],
    index_col=["id"],
    engine="pyarrow",
)

max_date = test["date"].max()
min_date = max_date - pd.DateOffset(years=2)

# Set global config variables for running session
config.set_config(
    SEASONAL_PERIOD=7,
    MIN_LENGTH=30,
    COUNTRY="EC",
    MIN_DATE=min_date,
    MAX_DATE=max_date,
    FH_SIZE=(max_date - test["date"].min()).days + 1,
)

from megatron.transformers import (  # noqa: E402
    InitialPreprocessing,
    Mapper,
    PlateauDetector,
)
from megatron.visualization import seriesPlot  # noqa: E402

px.defaults.width, px.defaults.height = config.FIG_WIDTH, config.FIG_HEIGHT  # type: ignore
pio.renderers.default = "png"

# Exogenous data

## Oil

In [ ]:
# Oil prices during both the train and test periods
# Frequency is also restored to daily, and missing values are linearly interpolated
oil = (
    pd.read_csv(
        filepath_or_buffer="../data/raw/oil.csv",
        index_col=["date"],
        parse_dates=["date"],
    )
    .asfreq("D")
    .loc[min_date:]
    .interpolate(method="linear")
    .rename(columns={"dcoilwtico": "oil"})
)
oil.head()

In [ ]:
seriesPlot(data=oil, title="Ecuador series")

## Stores

In [ ]:
# Per store metadata, only required fields are extracted
stores = (
    pd.read_csv("../data/raw/stores.csv", index_col=["store_nbr"])
    .drop(columns=["city", "state"])
    .astype({"cluster": float})
    .rename(columns={"cluster": "store_group"})
)
stores["type"] = stores["type"].factorize()[0].astype(float)
stores.head()

In [ ]:
stores["type"].value_counts()

In [ ]:
stores["store_group"].value_counts()

## Transactions

In [ ]:
# Transactions per store during only the train period
# Restored to daily frequency with no interpolation
transactions = (
    pd.read_csv(
        filepath_or_buffer="../data/raw/transactions.csv",
        index_col=["store_nbr", "date"],
        parse_dates=["date"],
        engine="pyarrow",
    )
    .groupby("store_nbr")
    .apply(lambda x: x.droplevel(["store_nbr"]).asfreq("D").loc[min_date:])
    .sort_index()
)
transactions.head()

### Initial filters

In [ ]:
# Initial processing with three consistent meaningful steps:
# 1. drop zero series - series with all zero values, which are useless;
# 2. trim leading zeros - zero values, which can be considered as
#    a no activity;
# 3. drop trailing zero window series - zero values sequence with a specific length
#    which demand absence in most recent period.
processing = InitialPreprocessing(w=2 * config.SEASONAL_PERIOD)  # type: ignore
transactions = processing.drop_zero_series(transactions)
transactions = processing.trim_leading_zeros(transactions)
transactions = processing.drop_trailing_zero_window_series(transactions)

### Plateau detection

In [ ]:
# Defining a temporal inactivity expressed as a plateau - zero values sequence with
# a specific length
seriesPlot(
    data=transactions,
    n_series=4,
    w=2 * config.SEASONAL_PERIOD,  # type: ignore
    pld=True,
    seed=12,
)

In [ ]:
# Leave only data which follows the last plateau if any defined
transactions = PlateauDetector(
    w=2 * config.SEASONAL_PERIOD,  # type: ignore
    truncate=True,
).fit_transform(transactions)

In [ ]:
instance = 18
seriesPlot(
    data=transactions.loc[instance],  # type: ignore
    w=2 * config.SEASONAL_PERIOD,  # type: ignore
    title=f"Transactions in Equador for {instance}",
)

# Sales

In [ ]:
# Sales per store and product family - the target variable
# Restored to daily frequency with no interpolation
sales = (
    pd.read_csv(
        filepath_or_buffer="../data/raw/train.csv",
        index_col=["store_nbr", "family", "date"],
        parse_dates=["date"],
        engine="pyarrow",
    )
    .groupby(["store_nbr", "family"])
    .apply(lambda x: x.droplevel(["store_nbr", "family"]).asfreq("D").loc[min_date:])
    .sort_index()
)

# Exogenous variables per store and product family for both train and test periods:
# 1. on promotion - the number of items on promotion, which can be considered as
#  a proxy for marketing activity and demand stimulation.
X_exog_train = (
    sales[["onpromotion"]]
    .fillna(0)
    .astype(float)
    .rename(columns={"onpromotion": "on_promotion"})
)
X_exog_test = (
    test.set_index(["store_nbr", "family", "date"])
    .astype(float)
    .rename(columns={"onpromotion": "on_promotion"})
    .sort_index()
)

sales = sales[["sales"]]
sales.head()

## Multiindex reduction

In [ ]:
# Replacing multiindex with a single index to avoid further pandas index manipulation
# issues
mapper = Mapper()
sales = mapper.fit_transform(sales)
sales.head()  # type: ignore

## Initial filters

In [ ]:
processing = InitialPreprocessing(w=2 * config.SEASONAL_PERIOD)  # type: ignore
sales = processing.drop_zero_series(sales)  # type: ignore
sales = processing.trim_leading_zeros(sales)
sales = processing.drop_trailing_zero_window_series(sales)

In [ ]:
sales = mapper.inverse_transform(sales)

## Plateau detection

In [ ]:
seriesPlot(
    data=sales.loc[[18, 25]].sort_index(),  # type: ignore
    n_series=4,
    w=2 * config.SEASONAL_PERIOD,  # type: ignore
    pld=True,
    pd_value=0,
    seed=22,
)

In [ ]:
# Leave only product data which matches its store activity
# If any plateau lies in further they'd be processed on transformation stage
sales = sales.join(transactions[[]], how="inner")  # type: ignore

In [ ]:
seriesPlot(
    data=sales.loc[[18, 25]].sort_index(),
    n_series=4,
    w=2 * config.SEASONAL_PERIOD,  # type: ignore
    pld=True,
    pd_value=0,
    seed=22,
)

# Saving

In [ ]:
# Aligning exogenous variables with the target variable by index
X_exog_train = X_exog_train.join(sales[[]], how="inner")
X_exog_test = X_exog_test.join(
    sales[~sales.droplevel(-1).index.duplicated(keep="first")][[]].droplevel(-1),
    how="inner",
)

In [ ]:
# Saving processed data to parquet format for better performance and easier load
oil.to_parquet("../data/processed/oil.pq")
stores.to_parquet("../data/processed/stores.pq")
transactions.to_parquet("../data/processed/transactions.pq")  # type: ignore
sales.to_parquet("../data/processed/sales.pq")
X_exog_train.to_parquet("../data/processed/sales_exog_train.pq")
X_exog_test.to_parquet("../data/processed/sales_exog_test.pq")